# Projeto Forecast — 02. Limpeza dos dados

Corrige as colunas com decimais corrompidos identificadas na ingestão (`01_ingestao.ipynb`), padroniza os nomes das colunas e salva a base limpa em `data/world_population_limpo.csv`.

In [1]:
import pandas as pd
from pathlib import Path

DATA = Path('data')
df = pd.read_csv(DATA / 'world_population.csv', sep=';', encoding='utf-8')
df.shape

(234, 17)

## 1. Densidade (per km²)

Os pontos decimais viraram separadores de milhar (`2.224.774` em vez de `222,4774`), então o valor original não é recuperável com segurança a partir do texto. Como densidade = população 2022 ÷ área, recalculamos a coluna.

In [2]:
densidade_calc = df['2022 Population'] / df['Area (km²)']

comparacao = pd.DataFrame({
    'Country/Territory': df['Country/Territory'],
    'original': df['Density (per km²)'],
    'recalculada': densidade_calc.round(4),
})
comparacao.head(10)

,Country/Territory,original,recalculada
0,Afghanistan,630.587,63.0587
1,Albania,988.702,98.8702
2,Algeria,188.531,18.8531
3,American Samoa,2.224.774,222.4774
4,Andorra,1.705.641,170.5641
5,Angola,285.466,28.5466
6,Anguilla,1.742.527,174.2527
7,Antigua and Barbuda,2.121.335,212.1335
8,Argentina,163.683,16.3683
9,Armenia,934.831,93.4831


In [3]:
df['Density (per km²)'] = densidade_calc.round(4)

## 2. Growth Rate

A taxa de crescimento é um fator anual próximo de 1 (ex.: `1.0257` = +2,57% ao ano). Os valores corrompidos aparecem 10× maiores (`10.257`). Não existe nenhum valor entre 1,1 e 9, então a separação é clara: valores ≥ 10 são divididos por 10.

In [4]:
gr = df['Growth Rate']
print('Valores >= 10:', (gr >= 10).sum())
print('Valores entre 1.1 e 9:', ((gr > 1.1) & (gr < 9)).sum())

df['Growth Rate'] = gr.where(gr < 10, gr / 10)
df['Growth Rate'].describe()

Valores >= 10: 171
Valores entre 1.1 e 9: 0


count    234.000000
mean       1.009577
std        0.013385
min        0.912000
25%        1.001775
50%        1.007900
75%        1.016950
max        1.069100
Name: Growth Rate, dtype: float64

## 3. World Population Percentage

A coluna está correta: a soma fica perto de 100% (a diferença vem do arredondamento).

In [5]:
df['World Population Percentage'].sum()

np.float64(99.93)

## 4. Nomes das colunas

Padroniza para `snake_case`, facilitando o uso nas próximas etapas.

In [6]:
df = df.rename(columns={
    'Rank': 'rank',
    'CCA3': 'cca3',
    'Country/Territory': 'pais',
    'Capital': 'capital',
    'Continent': 'continente',
    '2022 Population': 'pop_2022',
    '2020 Population': 'pop_2020',
    '2015 Population': 'pop_2015',
    '2010 Population': 'pop_2010',
    '2000 Population': 'pop_2000',
    '1990 Population': 'pop_1990',
    '1980 Population': 'pop_1980',
    '1970 Population': 'pop_1970',
    'Area (km²)': 'area_km2',
    'Density (per km²)': 'densidade_km2',
    'Growth Rate': 'taxa_crescimento',
    'World Population Percentage': 'pct_pop_mundial',
})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   rank              234 non-null    int64  
 1   cca3              234 non-null    object 
 2   pais              234 non-null    object 
 3   capital           234 non-null    object 
 4   continente        234 non-null    object 
 5   pop_2022          234 non-null    int64  
 6   pop_2020          234 non-null    int64  
 7   pop_2015          234 non-null    int64  
 8   pop_2010          234 non-null    int64  
 9   pop_2000          234 non-null    int64  
 10  pop_1990          234 non-null    int64  
 11  pop_1980          234 non-null    int64  
 12  pop_1970          234 non-null    int64  
 13  area_km2          234 non-null    int64  
 14  densidade_km2     234 non-null    float64
 15  taxa_crescimento  234 non-null    float64
 16  pct_pop_mundial   234 non-null    float64
dt

## 5. Validação e exportação

In [7]:
assert df['densidade_km2'].dtype == 'float64'
assert df['taxa_crescimento'].between(0.9, 1.1).all()
assert df.isna().sum().sum() == 0

df.to_csv(DATA / 'world_population_limpo.csv', index=False)
df.head()

,rank,cca3,pais,capital,continente,pop_2022,pop_2020,pop_2015,pop_2010,pop_2000,pop_1990,pop_1980,pop_1970,area_km2,densidade_km2,taxa_crescimento,pct_pop_mundial
0,36,AFG,Afghanistan,Kabul,Asia,41128771,38972230,33753499,28189672,19542982,10694796,12486631,10752971,652230,63.0587,1.0257,0.52
1,138,ALB,Albania,Tirana,Europe,2842321,2866849,2882481,2913399,3182021,3295066,2941651,2324731,28748,98.8702,0.9957,0.04
2,34,DZA,Algeria,Algiers,Africa,44903225,43451666,39543154,35856344,30774621,25518074,18739378,13795915,2381741,18.8531,1.0164,0.56
3,213,ASM,American Samoa,Pago Pago,Oceania,44273,46189,51368,54849,58230,47818,32886,27075,199,222.4774,0.9831,0.00
4,203,AND,Andorra,Andorra la Vella,Europe,79824,77700,71746,71519,66097,53569,35611,19860,468,170.5641,1.0100,0.00
